# Chapter 8: Claims — Decomposing Agent Output into Verifiable Units

**Learning objectives:**
- Understand the claim taxonomy (Fact, Threshold, Rule, Artifact, ToolUse, Semantic)
- Construct claims from mock agent output with full provenance
- Use the ClaimCompiler to extract typed claims automatically from DOE question metadata and agent answers

---
## Preamble

Chapters 4–7 built the core evaluation pipeline: ingest a document, generate DOE questions, score answers with the geometric judge and analyze factor effects. But the pipeline so far treats each agent answer as a single unit — correct or incorrect.

In practice, a single agent response may contain **multiple verifiable assertions** of different types. The sentence "Model MDL-001 is in Credit Decisioning and its test AUC of 0.8412 exceeds the 0.80 threshold" contains a factual triple, a numeric artifact and a threshold comparison — each requiring a different verification strategy.

This chapter introduces the **claim layer**: a typed decomposition that turns free-form agent output into discrete, verifiable units routable to specialized GMS verifiers.

Key modules:
- `knowlytix.harness.testing.claims` — `ClaimProvenance`, `ClaimBase`, `FactClaim`, `ThresholdClaim`, `RuleClaim`, `ArtifactClaim`, `ToolUseClaim`, `SemanticClaim`, `ClaimCompiler`
- `demo.mock_agent` — `MockHarness` (for tool-trace extraction)

In [ ]:
# === Preamble (all notebooks) ===
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
from collections import Counter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
from importlib.resources import files
# Load the Chapter 4 store
from knowlytix.harness.testing import DOEGMSBenchmark, DOEHarnessConfig

config = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md")),
    stores_dir="../lab_stores",
    store_name="ch04_ground_truth",
    n_runs=4,
    ingest_epochs=500,
    enable_tracing=False,
    device=str(DEVICE),
)

bench = DOEGMSBenchmark(config)
store = bench.ingest()
print(f"\nStore: {store.stats()['entities']} entities, "
      f"{store.stats()['triples']} triples, "
      f"{store.stats()['enm_entries']} ENM")

---

## The Claim Taxonomy

The GMSH claim layer defines six typed claims. Each maps to a specific GMS verification route:

| Claim Type | What It Captures | Verified Via |
|---|---|---|
| `FactClaim(h, r, t)` | A triple-shaped factual assertion | `score_triple()` |
| `ThresholdClaim` | A numeric comparison (metric ≥ limit) | `lookup_enm()` + `check_threshold()` |
| `RuleClaim` | Path consistency or logical implication | `check_holonomy()` + `tension_energy()` |
| `ArtifactClaim` | An exact numeric value from ENM | `lookup_enm()` + SHA-256 |
| `ToolUseClaim` | An agent tool call and its claimed result | `score_triple()` on tool relations |
| `SemanticClaim` | Free-text meaning vs ground truth | LLM-as-judge anchored by GMS evidence |

Every claim inherits from `ClaimBase`, which provides a unique ID, run/question linkage, raw text fragment and a `ClaimProvenance` record tracing the claim back to its source.

In [ ]:
# Import the full claim taxonomy
from knowlytix.harness.testing.claims import (
    ClaimProvenance, FactClaim, ThresholdClaim, RuleClaim,
    ArtifactClaim, ToolUseClaim, SemanticClaim,
    ClaimCompiler,
)

# Show the taxonomy
claim_types = [
    ("FactClaim", FactClaim, "(head, relation, tail) triple"),
    ("ThresholdClaim", ThresholdClaim, "entity.metric op limit"),
    ("RuleClaim", RuleClaim, "path consistency / logical rule"),
    ("ArtifactClaim", ArtifactClaim, "exact numeric from ENM"),
    ("ToolUseClaim", ToolUseClaim, "agent tool call + result"),
    ("SemanticClaim", SemanticClaim, "free-text semantic match"),
]

print(f"{'Type':<20s} {'claim_type field':<18s} {'Description'}")
print("-" * 65)
for name, cls, desc in claim_types:
    instance = cls()
    print(f"{name:<20s} {instance.claim_type:<18s} {desc}")

---

## Lab 8.1 — Manually Constructing Claims with Provenance

Before using the automated compiler, we construct each claim type by hand to understand its structure. Consider an agent that answers: *"Model MDL-001 is deployed in the Credit Decisioning domain. Its test AUC of 0.8412 exceeds the 0.80 regulatory threshold."*

This single sentence contains three verifiable assertions:
1. A **fact**: (MDL-001, domain, Credit Decisioning)
2. An **artifact**: test AUC = 0.8412
3. A **threshold**: test AUC ≥ 0.80

In [ ]:
# Lab 8.1a: Create ClaimProvenance — traces claim to source document
provenance = ClaimProvenance(
    doc_id="model_risk_assessment.md",
    section_tag="model_inventory",
    span_start=0,
    span_end=95,
    extraction_method="metadata",
)

print("ClaimProvenance fields:")
for field_name in ["doc_id", "section_tag", "span_start", "span_end", "extraction_method"]:
    print(f"  {field_name:<20s} = {getattr(provenance, field_name)!r}")

In [ ]:
# Lab 8.1b: FactClaim — a triple-shaped assertion
fact = FactClaim(
    head="MDL-001",
    relation="domain",
    tail="Credit Decisioning",
    raw_text="Model MDL-001 is deployed in the Credit Decisioning domain",
    confidence=1.0,
    provenance=provenance,
    source_run_id="run-demo",
    source_question_id="q-001",
)

print("FactClaim:")
print(f"  claim_id:    {fact.claim_id}")
print(f"  claim_type:  {fact.claim_type}")
print(f"  triple:      ({fact.head}, {fact.relation}, {fact.tail})")
print(f"  confidence:  {fact.confidence}")
print(f"  raw_text:    {fact.raw_text[:60]}...")
print(f"  provenance:  doc_id={fact.provenance.doc_id}, "
      f"method={fact.provenance.extraction_method}")

In [ ]:
# Lab 8.1c: ArtifactClaim — an exact numeric value from ENM
artifact = ArtifactClaim(
    category="test_auc",
    entity_id="MDL-001",
    claimed_value=0.8412,
    raw_text="Its test AUC of 0.8412",
    confidence=1.0,
    provenance=ClaimProvenance(
        doc_id="model_risk_assessment.md",
        section_tag="model_inventory",
        extraction_method="numeric_match",
    ),
    source_run_id="run-demo",
    source_question_id="q-001",
)

print("ArtifactClaim:")
print(f"  claim_id:      {artifact.claim_id}")
print(f"  claim_type:    {artifact.claim_type}")
print(f"  category:      {artifact.category}")
print(f"  entity_id:     {artifact.entity_id}")
print(f"  claimed_value: {artifact.claimed_value}")
print(f"  confidence:    {artifact.confidence}")

In [ ]:
# Lab 8.1d: ThresholdClaim — a numeric comparison
threshold = ThresholdClaim(
    entity="MDL-001",
    metric="test_auc",
    value=0.8412,
    limit=0.80,
    operator="ge",
    raw_text="test AUC of 0.8412 exceeds the 0.80 regulatory threshold",
    confidence=1.0,
    provenance=ClaimProvenance(
        doc_id="model_risk_assessment.md",
        section_tag="model_inventory",
        extraction_method="metadata",
    ),
    source_run_id="run-demo",
    source_question_id="q-001",
)

print("ThresholdClaim:")
print(f"  claim_id:  {threshold.claim_id}")
print(f"  claim_type: {threshold.claim_type}")
print(f"  entity:    {threshold.entity}")
print(f"  metric:    {threshold.metric}")
print(f"  assertion: {threshold.value} {threshold.operator} {threshold.limit}")
print(f"  human:     test_auc = {threshold.value} >= {threshold.limit} ✓")

In [ ]:
# Lab 8.1e: RuleClaim — path consistency
rule = RuleClaim(
    antecedent_entities=["MDL-001", "Credit Decisioning", "Regulatory Review"],
    antecedent_relations=["domain", "requires"],
    consequent_entity="Regulatory Review",
    direct_relation="subject_to",
    rule_type="consistency",
    raw_text="MDL-001 is in Credit Decisioning which requires Regulatory Review",
    confidence=1.0,
    provenance=ClaimProvenance(
        doc_id="model_risk_assessment.md",
        section_tag="governance",
        extraction_method="metadata",
    ),
    source_run_id="run-demo",
    source_question_id="q-002",
)

print("RuleClaim:")
print(f"  claim_id:   {rule.claim_id}")
print(f"  claim_type: {rule.claim_type}")
print(f"  rule_type:  {rule.rule_type}")
print(f"  path:       {' -> '.join(rule.antecedent_entities)}")
print(f"  relations:  {rule.antecedent_relations}")
print(f"  consequent: {rule.consequent_entity}")
print(f"  direct_rel: {rule.direct_relation}")

In [ ]:
# Lab 8.1f: ToolUseClaim — from an agent tool call
tool_claim = ToolUseClaim(
    tool_name="risk_check",
    tool_args={"model_id": "MDL-001", "check_type": "threshold"},
    result_key="risk_level",
    claimed_value="Medium",
    raw_text="Agent called risk_check(model_id=MDL-001) -> risk_level=Medium",
    confidence=1.0,
    provenance=ClaimProvenance(
        doc_id="tool_trace",
        section_tag="tool_call_3",
        extraction_method="metadata",
    ),
    source_run_id="run-demo",
    source_question_id="q-003",
)

print("ToolUseClaim:")
print(f"  claim_id:      {tool_claim.claim_id}")
print(f"  claim_type:    {tool_claim.claim_type}")
print(f"  tool_name:     {tool_claim.tool_name}")
print(f"  tool_args:     {tool_claim.tool_args}")
print(f"  result_key:    {tool_claim.result_key}")
print(f"  claimed_value: {tool_claim.claimed_value}")

In [ ]:
# Lab 8.1g: SemanticClaim — free-text meaning match
semantic = SemanticClaim(
    ground_truth="Credit Decisioning",
    question_text="What domain is MDL-001 deployed in?",
    context_section="model_inventory",
    semantic_type="equivalence",
    raw_text="The model operates within the credit decisioning space",
    confidence=0.8,
    provenance=ClaimProvenance(
        doc_id="model_risk_assessment.md",
        section_tag="model_inventory",
        extraction_method="semantic",
    ),
    source_run_id="run-demo",
    source_question_id="q-001",
)

print("SemanticClaim:")
print(f"  claim_id:      {semantic.claim_id}")
print(f"  claim_type:    {semantic.claim_type}")
print(f"  semantic_type: {semantic.semantic_type}")
print(f"  ground_truth:  {semantic.ground_truth}")
print(f"  question:      {semantic.question_text}")
print(f"  confidence:    {semantic.confidence}")

In [ ]:
# Summary: all six claims from one agent response
all_manual = [fact, artifact, threshold, rule, tool_claim, semantic]

print("Claims extracted from a single agent response:")
print(f"{'#':<4s} {'Type':<16s} {'Confidence':>10s} {'Extraction':>14s} {'Key Info'}")
print("-" * 80)
for i, c in enumerate(all_manual, 1):
    method = c.provenance.extraction_method if c.provenance else "n/a"
    if c.claim_type == "fact":
        info = f"({c.head}, {c.relation}, {c.tail})"
    elif c.claim_type == "artifact":
        info = f"{c.category}({c.entity_id}) = {c.claimed_value}"
    elif c.claim_type == "threshold":
        info = f"{c.metric} = {c.value} {c.operator} {c.limit}"
    elif c.claim_type == "rule":
        info = f"{c.rule_type}: {' -> '.join(c.antecedent_entities)}"
    elif c.claim_type == "tool_use":
        info = f"{c.tool_name}() -> {c.result_key}={c.claimed_value}"
    elif c.claim_type == "semantic":
        info = f"{c.semantic_type}: vs '{c.ground_truth}'"
    else:
        info = c.raw_text[:40]
    print(f"{i:<4d} {c.claim_type:<16s} {c.confidence:>10.1f} {method:>14s} {info}")

Each claim type maps to a different GMS verification strategy. The `claim_type` field acts as a routing key — Chapter 9 will show how verifiers use this to select the correct GMS primitive.

---

## ClaimProvenance: Tracing Claims to Source

Every claim carries a `ClaimProvenance` record that links it back to its origin. This is essential for auditability in regulated environments — when a verification fails, stakeholders need to trace the failure to the exact document section and extraction method.

The three extraction methods correspond to different confidence levels:

| Method | Confidence | Description |
|---|---|---|
| `metadata` | 1.0 | Claim derived from structured question metadata |
| `text_match` | 0.6 | Entity pair found in answer text, matched to known triple |
| `numeric_match` | 0.5 | Number in answer matched to ENM entry |
| `semantic` | 0.8 | Free-text meaning compared via LLM judge |

In [ ]:
# Demonstrate provenance differences across extraction methods
methods = {
    "metadata": ClaimProvenance(
        doc_id="model_risk_assessment.md",
        section_tag="model_inventory",
        span_start=0, span_end=45,
        extraction_method="metadata",
    ),
    "text_match": ClaimProvenance(
        doc_id="model_risk_assessment.md",
        section_tag="model_inventory",
        span_start=12, span_end=38,
        extraction_method="text_match",
    ),
    "numeric_match": ClaimProvenance(
        doc_id="model_risk_assessment.md",
        section_tag="performance_metrics",
        span_start=55, span_end=61,
        extraction_method="numeric_match",
    ),
}

confidences = {"metadata": 1.0, "text_match": 0.6, "numeric_match": 0.5}

print(f"{'Method':<16s} {'Confidence':>10s} {'Section':<22s} {'Span'}")
print("-" * 65)
for name, prov in methods.items():
    print(f"{name:<16s} {confidences[name]:>10.1f} {prov.section_tag:<22s} "
          f"[{prov.span_start}, {prov.span_end})")

---

## The ClaimCompiler: Automated Extraction

Manually constructing claims is useful for understanding the data model, but in practice the `ClaimCompiler` automates extraction. It applies a three-tier strategy:

1. **Metadata-guided** (confidence 1.0): Question metadata contains structured fields (head/relation/tail, encoder/limit/op, enm_type/entity_id) that map directly to typed claims.
2. **Text-match** (confidence 0.6): Scan the answer for entity pairs known to the GMS store and emit FactClaims for matching triples.
3. **Numeric-match** (confidence 0.5): Parse numbers from the answer and match against ENM entries.

Let us generate DOE questions and run the compiler on real question metadata.

In [ ]:
# Generate DOE questions
doe_questions = bench.generate_questions()
print(f"Generated {len(doe_questions)} DOE questions across "
      f"{len(set(dq.question.category for dq in doe_questions))} categories")

In [ ]:
# Create the ClaimCompiler with our GMS store
compiler = ClaimCompiler(store=store)

# Compile claims from the first 5 DOE questions using metadata
print("Compiling claims from the first 5 DOE questions:\n")
for i, dq in enumerate(doe_questions[:5]):
    q = dq.question
    # Simulate an agent answer (use ground truth for demonstration)
    agent_answer = str(q.ground_truth)

    claims = compiler.compile(
        llm_answer=agent_answer,
        question_metadata=q.metadata,
        question_category=q.category,
        run_id="demo-run",
        question_id=f"q-{i:03d}",
        ground_truth=str(q.ground_truth),
        question_text=q.natural_language,
    )

    print(f"Q{i}: category={q.category}")
    print(f"    answer: {agent_answer[:70]}{'...' if len(agent_answer) > 70 else ''}")
    print(f"    claims extracted: {len(claims)}")
    for c in claims:
        method = c.provenance.extraction_method if c.provenance else "n/a"
        print(f"      [{c.claim_type:<12s}] conf={c.confidence:.1f} "
              f"method={method}")
    print()

In [ ]:
# Compile claims from ALL 640 DOE questions
all_claims = []
claims_per_question = []

for i, dq in enumerate(doe_questions):
    q = dq.question
    agent_answer = str(q.ground_truth)  # perfect agent for claim coverage

    claims = compiler.compile(
        llm_answer=agent_answer,
        question_metadata=q.metadata,
        question_category=q.category,
        run_id="demo-run",
        question_id=f"q-{i:03d}",
        ground_truth=str(q.ground_truth),
        question_text=q.natural_language,
    )
    all_claims.extend(claims)
    claims_per_question.append(len(claims))

print(f"Total claims extracted: {len(all_claims)} from {len(doe_questions)} questions")
print(f"Claims per question: min={min(claims_per_question)}, "
      f"max={max(claims_per_question)}, "
      f"mean={np.mean(claims_per_question):.1f}")

In [ ]:
# Distribution by claim type
type_counts = Counter(c.claim_type for c in all_claims)

print(f"\n{'Claim Type':<16s} {'Count':>7s} {'Pct':>7s}")
print("-" * 34)
for ctype, count in type_counts.most_common():
    pct = count / len(all_claims) * 100
    print(f"{ctype:<16s} {count:>7d} {pct:>6.1f}%")

# Distribution by extraction method
method_counts = Counter(
    c.provenance.extraction_method if c.provenance else "none"
    for c in all_claims
)

print(f"\n{'Extraction Method':<18s} {'Count':>7s} {'Pct':>7s}")
print("-" * 36)
for method, count in method_counts.most_common():
    pct = count / len(all_claims) * 100
    print(f"{method:<18s} {count:>7d} {pct:>6.1f}%")

In [ ]:
# Claims per question category
cat_claims = {}
idx = 0
for i, dq in enumerate(doe_questions):
    cat = dq.question.category
    n = claims_per_question[i]
    if cat not in cat_claims:
        cat_claims[cat] = []
    cat_claims[cat].append(n)

print(f"{'Category':<25s} {'Questions':>9s} {'Claims':>7s} {'Mean':>6s} {'Max':>5s}")
print("-" * 56)
for cat in sorted(cat_claims.keys()):
    counts = cat_claims[cat]
    print(f"{cat:<25s} {len(counts):>9d} {sum(counts):>7d} "
          f"{np.mean(counts):>5.1f} {max(counts):>5d}")

The claim distribution reveals the verification profile of the evaluation. Categories like `workflow_compliance` and `multi_hop` produce multiple claims per question (path hops emit individual FactClaims plus a RuleClaim for path consistency), while `exact_retrieval` questions typically produce a single FactClaim.

---

## Inspecting Claim Structure

Let us pick a multi-hop question that produces several claims and inspect each one in detail.

In [ ]:
# Find a multi-hop or workflow_compliance question with multiple claims
rich_idx = None
rich_claims = []
for i, dq in enumerate(doe_questions):
    if dq.question.category in ("procedural", "holonomy", "multi_hop_transport"):
        q = dq.question
        claims = compiler.compile(
            llm_answer=str(q.ground_truth),
            question_metadata=q.metadata,
            question_category=q.category,
            run_id="demo-run",
            question_id=f"q-{i:03d}",
            ground_truth=str(q.ground_truth),
            question_text=q.natural_language,
        )
        if len(claims) >= 3:
            rich_idx = i
            rich_claims = claims
            break

if rich_idx is not None:
    dq = doe_questions[rich_idx]
    q = dq.question
    print(f"Question {rich_idx}: {q.category}")
    print(f"  Text: {q.natural_language[:100]}...")
    print(f"  Ground truth: {q.ground_truth}")
    print(f"  Metadata keys: {list(q.metadata.keys())}")
    print(f"\n  Claims ({len(rich_claims)}):\n")
    for j, c in enumerate(rich_claims):
        print(f"  --- Claim {j+1}: {c.claim_type} ---")
        if c.claim_type == "fact":
            print(f"    triple: ({c.head}, {c.relation}, {c.tail})")
        elif c.claim_type == "rule":
            print(f"    path: {' -> '.join(c.antecedent_entities)}")
            print(f"    relations: {c.antecedent_relations}")
            print(f"    rule_type: {c.rule_type}")
        elif c.claim_type == "artifact":
            print(f"    {c.category}({c.entity_id}) = {c.claimed_value}")
        elif c.claim_type == "threshold":
            print(f"    {c.entity}.{c.metric} {c.operator} {c.limit}")
        print(f"    confidence: {c.confidence}")
        print(f"    provenance: {c.provenance.extraction_method}")
else:
    print("No multi-claim question found (unexpected)")

The multi-hop question decomposes into individual hop FactClaims plus a RuleClaim for path consistency. Each claim can be verified independently — a FactClaim via `score_triple()` and a RuleClaim via `check_holonomy()` and `tension_energy()`. This is the power of typed decomposition: one question becomes multiple verification tasks, each with a deterministic GMS route.

---

## Lab 8.2 — Claims from Mock Agent Tool Traces

When testing agentic systems, the agent's **tool calls** are themselves verifiable claims. If an agent calls `risk_check(model_id="MDL-001")` and reports the result, we can construct a `ToolUseClaim` and verify the claimed result against the GMS store.

We run the `MockHarness`, inspect its tool-call records and construct claims from the trace.

In [ ]:
# Lab 8.2a: Run MockHarness and inspect tool records
from knowlytix.harness.fixtures import MockHarness

harness = MockHarness()
prompt = "What is the risk level for model MDL-001 in Credit Decisioning?"
answer = harness.run(prompt)

print(f"Prompt:  {prompt}")
print(f"Answer:  {answer[:120]}{'...' if len(answer) > 120 else ''}")
print(f"\nTool calls recorded: {len(harness._tool_records)}")
for i, rec in enumerate(harness._tool_records):
    blocked = " [BLOCKED]" if rec.get("blocked", False) else ""
    print(f"  {i+1}. {rec['tool']}({rec['args']}){blocked}")
    print(f"     result: {str(rec['result'])[:80]}")
    print()

In [ ]:
# Lab 8.2b: Construct ToolUseClaims from tool records
tool_claims = []
for i, rec in enumerate(harness._tool_records):
    claim = ToolUseClaim(
        tool_name=rec["tool"],
        tool_args=rec["args"],
        result_key="result",
        claimed_value=str(rec["result"])[:200],
        raw_text=f"{rec['tool']}({rec['args']}) -> {str(rec['result'])[:100]}",
        confidence=1.0,
        provenance=ClaimProvenance(
            doc_id="tool_trace",
            section_tag=f"tool_call_{i+1}",
            extraction_method="metadata",
        ),
        source_run_id="mock-run-001",
        source_question_id="q-mock-001",
    )
    tool_claims.append(claim)

print(f"Constructed {len(tool_claims)} ToolUseClaim(s) from mock agent trace:\n")
for c in tool_claims:
    print(f"  [{c.claim_type}] {c.tool_name}")
    print(f"    args:     {c.tool_args}")
    print(f"    claimed:  {c.claimed_value[:80]}")
    print(f"    prov:     {c.provenance.section_tag}, "
          f"method={c.provenance.extraction_method}")
    print()

In [ ]:
# Lab 8.2c: Construct FactClaims from the agent's natural-language answer
# Use the text-match tier of ClaimCompiler
answer_claims = compiler.compile(
    llm_answer=answer,
    question_metadata={},  # no metadata — rely on text/numeric match
    question_category="agent_response",
    run_id="mock-run-001",
    question_id="q-mock-001",
    ground_truth="",
    question_text=prompt,
)

print(f"Claims extracted from agent answer text ({len(answer_claims)}):\n")
if answer_claims:
    for c in answer_claims:
        method = c.provenance.extraction_method if c.provenance else "n/a"
        if c.claim_type == "fact":
            print(f"  [fact] ({c.head}, {c.relation}, {c.tail}) "
                  f"conf={c.confidence} method={method}")
        elif c.claim_type == "artifact":
            print(f"  [artifact] {c.category}({c.entity_id}) = {c.claimed_value} "
                  f"conf={c.confidence} method={method}")
        else:
            print(f"  [{c.claim_type}] conf={c.confidence} method={method}")
else:
    print("  (No claims extracted — mock agent answer may not contain")
    print("   recognizable entity names from the GMS store)")

In [ ]:
# Lab 8.2d: Combine tool-trace and answer-text claims
combined = tool_claims + answer_claims

print(f"Combined claim set: {len(combined)} claims")
print(f"  From tool trace:  {len(tool_claims)}")
print(f"  From answer text: {len(answer_claims)}")
print()

# All claims carry provenance — verify
has_provenance = sum(1 for c in combined if c.provenance is not None)
print(f"Claims with provenance: {has_provenance}/{len(combined)}")

# Show provenance chain for audit
if combined:
    print("\nProvenance audit trail:")
    print(f"{'#':<4s} {'Type':<14s} {'Method':<16s} {'Section':<18s} {'Doc ID'}")
    print("-" * 72)
    for i, c in enumerate(combined, 1):
        if c.provenance:
            print(f"{i:<4d} {c.claim_type:<14s} {c.provenance.extraction_method:<16s} "
                  f"{c.provenance.section_tag:<18s} {c.provenance.doc_id}")
        else:
            print(f"{i:<4d} {c.claim_type:<14s} {'n/a':<16s} {'n/a':<18s} n/a")

The mock agent's tool trace provides high-confidence claims (the agent definitively called specific tools), while the text-match extraction of the answer yields lower-confidence claims that need store verification. Together they form a complete claim set ready for the verification pipeline in Chapter 9.

---

## Exercises

### Exercise 8.1

Construct a `ThresholdClaim` for the assertion "MDL-001's Gini coefficient of 0.72 falls below the 0.75 minimum." Set the operator to `"ge"` (greater-or-equal). Does the claimed value (0.72) satisfy the threshold? What would a verifier need to check?

### Exercise 8.2

Run the `ClaimCompiler` on the first 100 DOE questions but pass an **incorrect** answer (e.g., always "Unknown") instead of the ground truth. Compare the claim distribution (by type and extraction method) to the distribution from correct answers. Which extraction tiers are affected and why?

### Exercise 8.3

Extend Lab 8.2 by running the `MockHarness` on three different prompts. For each run, extract tool-trace claims and answer-text claims. Build a combined DataFrame with columns `[run_id, claim_type, tool_name, confidence, extraction_method]` and show the claim counts per run.

In [ ]:
# === Exercise 8.1: Your code here ===
# Hint:
# tc = ThresholdClaim(
#     entity="MDL-001", metric="gini_coefficient",
#     value=0.72, limit=0.75, operator="ge",
#     raw_text="MDL-001's Gini coefficient of 0.72 falls below the 0.75 minimum",
#     confidence=1.0,
#     provenance=ClaimProvenance(extraction_method="metadata"),
# )
# Does 0.72 >= 0.75? No — the claim would FAIL verification.

In [ ]:
# === Exercise 8.2: Your code here ===
# Hint:
# wrong_claims = []
# for i, dq in enumerate(doe_questions[:100]):
#     claims = compiler.compile(
#         llm_answer="Unknown",
#         question_metadata=dq.question.metadata,
#         question_category=dq.question.category,
#         run_id="wrong-run", question_id=f"q-{i:03d}",
#     )
#     wrong_claims.extend(claims)
# Compare: metadata claims should be identical, text/numeric should differ

In [ ]:
# === Exercise 8.3: Your code here ===
# Hint: run harness on 3 prompts, collect tool_claims + answer_claims per run

---

## Summary

This chapter introduced the claim layer — the bridge between free-form agent output and deterministic GMS verification:

1. **Six typed claims** (Fact, Threshold, Rule, Artifact, ToolUse, Semantic) decompose agent output into discrete, verifiable units. Each type maps to a specific GMS verification route.
2. **ClaimProvenance** traces every claim back to its source document, section and extraction method — essential for auditability in regulated environments.
3. **ClaimCompiler** automates extraction via a three-tier strategy: metadata-guided (confidence 1.0), text-match (confidence 0.6) and numeric-match (confidence 0.5).
4. **Tool-trace claims** capture agent tool calls as ToolUseClaims, providing high-confidence evidence of what the agent actually did versus what it reported.
5. **Multi-hop decomposition** turns complex questions into multiple independent claims — one per hop plus a path-consistency RuleClaim.

**Next:** Chapter 9 introduces **verifiers** — specialized modules that route each claim type to the correct GMS primitive and produce `SubVerdict` objects with pass/fail status, confidence scores and failure codes.